# Gradient descent, loss functions (cross-entropy)

*0.2 Math / ML basics · run **Setup** first*

## Setup

Settings, a configured client, an `embed()` helper and numpy. Every cell below uses them.

In [1]:
"""Shared setup: typed settings, a configured client, numpy, and a print helper."""

import json

import numpy as np
from dotenv import find_dotenv
from openai import OpenAI
from pydantic import Field, SecretStr
from pydantic_settings import BaseSettings, SettingsConfigDict

np.set_printoptions(precision=4, suppress=True)


class Settings(BaseSettings):
    model_config = SettingsConfigDict(env_file=find_dotenv(), extra="ignore")

    openai_api_key: SecretStr
    openai_model: str = "gpt-4o-mini"
    embedding_model: str = "text-embedding-3-small"
    request_timeout_seconds: float = Field(default=30, gt=0)
    max_retries: int = Field(default=2, ge=0, le=5)


settings = Settings()
client = OpenAI(
    api_key=settings.openai_api_key.get_secret_value(),
    timeout=settings.request_timeout_seconds,
    max_retries=settings.max_retries,
)


def embed(texts: list[str]) -> np.ndarray:
    """Embed texts with the configured model; returns one row per text."""
    response = client.embeddings.create(model=settings.embedding_model, input=texts)
    rows = []
    for item in response.data:
        rows.append(item.embedding)
    return np.array(rows)


def show(title: str, value) -> None:
    print(title)
    print(json.dumps(value, indent=2, ensure_ascii=False, default=str))


print("chat model:", settings.openai_model, "| embedding model:", settings.embedding_model)

chat model: gpt-4o-mini | embedding model: text-embedding-3-small


### Gradient descent

> **Problem.** A model has millions of weights and a loss that says how wrong it is. There is no formula that gives the right weights directly; the only way is to nudge them repeatedly in the direction that reduces the loss.

**Idea.** Compute the slope of the loss with respect to each weight, step a little downhill, repeat.

**Use when** training or fine-tuning anything (layers 1.2–1.4); understanding why learning rate matters.  
**Not when** linear problems with a closed form — sklearn solves those exactly.

```mermaid
flowchart LR
    F[forward: prediction] --> L[loss] --> B[backward: gradients] --> S[optimizer.step: w ← w − lr·grad] --> F
```

**How it works.**
1. `w` and `b` are tensors with `requires_grad=True`; torch records every operation on them.
2. The forward pass computes the mean squared error between `w·x + b` and `y`.
3. `loss.backward()` runs autograd: it fills `w.grad` and `b.grad` with the slope of the loss.
4. `optimizer.step()` moves each weight against its gradient by `lr`; `zero_grad()` clears the slopes before the next round.
5. Too small a learning rate crawls (0.01 is still far after 200 steps); too large diverges to infinity (2.5); 0.1 lands on w≈3, b≈−2.

| | what happens | result |
|:--|:--|:--|
| ✗ lr=0.01 | 200 steps | still far from the answer |
| ✓ lr=0.1 | 200 steps | w≈3.0, b≈−2.0 |
| ✗ lr=2.5 | overshoots | diverged |

**Production code and its real output**

In [2]:
# Gradient descent — torch autograd computes the gradient; an optimizer steps down it.
import torch

torch.manual_seed(0)
x = torch.linspace(-1, 1, 200)
y = 3 * x - 2 + 0.1 * torch.randn(200)  # true line: w=3, b=-2, plus noise

w = torch.zeros(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
optimizer = torch.optim.SGD([w, b], lr=0.1)
for step in range(200):
    loss = torch.mean((w * x + b - y) ** 2)  # mean squared error
    optimizer.zero_grad()
    loss.backward()  # autograd: d loss / d w, d loss / d b
    optimizer.step()  # w ← w − lr · gradient
    if step in (0, 5, 20, 199):
        print(f"step {step:>3}  loss {loss.item():.4f}  w {w.item():.3f}  b {b.item():.3f}")

# Learning rate: too high diverges, too low crawls.
for lr in [0.01, 0.1, 2.5]:
    w2, b2 = torch.zeros(1, requires_grad=True), torch.zeros(1, requires_grad=True)
    opt = torch.optim.SGD([w2, b2], lr=lr)
    for _ in range(200):
        loss2 = torch.mean((w2 * x + b2 - y) ** 2)
        opt.zero_grad()
        loss2.backward()
        opt.step()
    print(
        
            f"lr={lr:<5} final loss "
            f"{'diverged' if not torch.isfinite(loss2) else round(loss2.item(), 4)}"
        
    )
assert abs(w.item() - 3) < 0.1 and abs(b.item() + 2) < 0.1

step   0  loss 7.0161  w 0.201  b -0.400
step   5  loss 1.9327  w 1.020  b -1.477
step  20  loss 0.1942  w 2.294  b -1.984
step 199  loss 0.0092  w 2.984  b -2.002
lr=0.01  final loss 0.2142
lr=0.1   final loss 0.0092
lr=2.5   final loss diverged


**What the output shows.** Loss fell from ≈5 toward the noise floor and the weights converged on the true line; the learning-rate sweep showed the crawl / converge / diverge pattern.

**In practice**
- **Adam, not SGD** — production training uses Adam/AdamW, which scales each weight's step by its gradient history — far less learning-rate tuning.
- **schedules** — warm up the learning rate for the first steps, then decay it (cosine); a constant rate is rarely optimal.
- **zero_grad every step** — gradients accumulate by default; forgetting `zero_grad()` sums them across steps.
- **mini-batches** — compute the gradient on a batch of examples, not the whole dataset — noisier but far cheaper per step.
- **gradient clipping** — cap the gradient norm (e.g. 1.0) so one bad batch cannot blow up training.

**Alternatives** — closed-form solvers for linear models (sklearn) · second-order methods (L-BFGS) for small problems

**Terms** — *gradient*: the slope of the loss for each weight · *learning rate*: how far to step each time · *autograd*: torch's automatic gradient calculation


### loss functions (cross-entropy)

> **Problem.** Training needs one number that says how wrong a prediction is — and for a next-token prediction over 100,000 candidates, "right or wrong" is not enough; being 90% sure of the right token must score better than being 30% sure.

**Idea.** Cross-entropy is −log of the probability the model gave the correct answer: confident-and-right ≈ 0, confident-and-wrong is huge.

**Use when** every classification and language-model training run.  
**Not when** regression (continuous targets) — use mean squared error.

```
p(correct) = 0.95  →  −log 0.95 = 0.05    confident & right
p(correct) = 0.37  →  −log 0.37 = 1.00    unsure
p(correct) = 0.02  →  −log 0.02 = 3.85    confident & wrong  ← punished hardest

perplexity = exp(mean cross-entropy)  ≈ "how many tokens the model was choosing between"
```

**How it works.**
1. `F.cross_entropy(logits, target)` applies log-softmax to the raw scores and returns −log p of the target class, averaged over the batch.
2. Three example logit vectors show the scale: 0.05 when confident and right, 1.0 when unsure, 3.85 when confident and wrong.
3. For language models the same loss is computed at every position over the vocabulary; the mean is what training curves plot.
4. `exp(mean loss)` is perplexity — the number reported in papers; 3.3 means the model was effectively choosing among ~3 tokens.

| | what happens | result |
|:--|:--|:--|
| ✓ right, confident | p=0.95 | loss 0.05 |
| ✗ wrong, confident | p=0.02 | loss 3.85 |
| ✓ perplexity | mean loss 1.2 | ≈3.3 |

**Production code and its real output**

In [3]:
# Cross-entropy — the loss for classification and language modelling: −log p(correct class).
# torch.nn.functional.cross_entropy takes raw logits and does log-softmax + pick + mean.
import torch
import torch.nn.functional as F

correct = torch.tensor([0])  # class 0 = "cat"
for name, logits in {
    "confident & right": [4.0, 0.5, 0.2],
    "unsure": [1.0, 0.9, 0.8],
    "confident & wrong": [0.2, 4.0, 0.5],
}.items():
    loss = F.cross_entropy(torch.tensor([logits]), correct)
    print(
        
            f"{name:<20} p(cat)={torch.softmax(torch.tensor(logits), 0)[0]:.3f}  "
            f"cross-entropy={loss.item():.3f}"
        
    )

# Language modelling: next-token classification over the vocabulary. Perplexity = exp(mean loss).
token_losses = torch.tensor([0.51, 1.20, 0.11, 3.00])  # −log p(correct next token) at 4 positions
print(
    "mean cross-entropy:",
    round(token_losses.mean().item(), 3),
    "| perplexity:",
    round(token_losses.mean().exp().item(), 2),
)
assert F.cross_entropy(torch.tensor([[4.0, 0.5, 0.2]]), correct) < F.cross_entropy(
    torch.tensor([[0.2, 4.0, 0.5]]), correct
)

confident & right    p(cat)=0.950  cross-entropy=0.051
unsure               p(cat)=0.367  cross-entropy=1.002
confident & wrong    p(cat)=0.021  cross-entropy=3.851
mean cross-entropy: 1.205 | perplexity: 3.34


**What the output shows.** The three cases produced 0.05, 1.0 and 3.85; the sequence example gave mean cross-entropy 1.2 and perplexity 3.3.

**In practice**
- **logits in, not probabilities** — the function does its own log-softmax; feeding it softmax output caps the gradient and stalls training (the 0.4 break→fix).
- **label smoothing** — training on soft targets (0.9/0.1 instead of 1/0) improves calibration and generalisation.
- **perplexity is model-specific** — compare perplexities only across models with the same tokenizer.
- **class imbalance** — weight the loss per class or the majority class dominates.

**Alternatives** — mean squared error (regression) · focal loss (extreme imbalance) · contrastive losses for embeddings (layer 1.3)

**Terms** — *cross-entropy*: −log p(correct) · *perplexity*: exp of the mean cross-entropy · *target*: the correct class or token
